## 4. Оцінка результатів

Перевірка `best.pt` на test, збір прикладів predict і відстеження, де модель сипиться

### Валідація на test

Після train прогняємо найкращі ваги на test-split, 1610 зображень, якого модель не бачила під час навчання

In [1]:
from ultralytics import YOLO

model = YOLO(r"../runs/detect/runs/detect/visdrone_yolov8n/weights/best.pt")
metrics = model.val(data="../data/visdrone.yaml", split="test")

Ultralytics 8.4.90  Python-3.11.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
Model summary (fused): 73 layers, 3,007,793 parameters, 0 gradients, 8.1 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)
                   all       1610      75102      0.392      0.291      0.259      0.144
            pedestrian       1197      21006      0.412       0.23      0.206     0.0761
                people        797       6376      0.455      0.096        0.1     0.0324
               bicycle        377       1302      0.243     0.0538     0.0527     0.0203
                   car       1530      28074       0.58      0.695      0.658      0.394
                   van       1168       5771      0.338      0.343       0.28      0.174
                 truck        750       2659       0.36      0.366      0.314      0.195
              tricycle        245        530      0.251        0.2       0.14     0.0703
       awning-

### Що показав test

Загалом: *mAP50 0.26*, *precision 0.39*, *recall 0.29*
трохи нижче ніж на val, але в тому ж діапазоні тримається

**car** знову тягне метрики (mAP50 *0.66*), *bus* теж ок (0.49);
Найгірше показали результати **bicycle** (0.05), **people** (0.10) і **awning-tricycle** (0.13), де мало прикладів, дрібні об'єкти, те саме що і показувала EDA.

### Приклади predict

Запускаю predict на кількох test-фото (для звіту достатньо 5). Відбираються вдалі в `assets/predictions/good/` і невдалі в `bad/`

In [3]:
from pathlib import Path

SAMPLE_IMAGES = [
    "../data/datasets/VisDrone/images/test/0000006_00159_d_0000001.jpg",
    "../data/datasets/VisDrone/images/test/0000011_00234_d_0000001.jpg",
    "../data/datasets/VisDrone/images/test/0000063_00500_d_0000001.jpg",
    "../data/datasets/VisDrone/images/test/0000011_01955_d_0000004.jpg",
    "../data/datasets/VisDrone/images/test/0000054_00786_d_0000001.jpg",
]

model.predict(
    source=SAMPLE_IMAGES,
    save=True,
    conf=0.25,
    project="../assets/predictions",
    name="samples",
    exist_ok=True,
)

print("Saved to:", Path("../assets/predictions/samples").resolve())


0: 640x640 37 cars, 2 vans, 12 trucks, 42.6ms
1: 640x640 23 pedestrians, 4 peoples, 3 cars, 1 van, 4 tricycles, 4 awning-tricycles, 1 motor, 42.6ms
2: 640x640 165 pedestrians, 1 people, 1 bicycle, 42.6ms
3: 640x640 23 pedestrians, 2 peoples, 1 car, 8 tricycles, 16 motors, 42.6ms
4: 640x640 71 pedestrians, 1 car, 4 motors, 42.6ms
Speed: 9.9ms preprocess, 42.6ms inference, 1.7ms postprocess per image at shape (1, 3, 640, 640)
Results saved to C:\Users\Artem\Documents\My_project\Trainee_test_Visicom\visdrone-yolov8-detection\runs\assets\predictions\samples
Saved to: C:\Users\Artem\Documents\My_project\Trainee_test_Visicom\visdrone-yolov8-detection\assets\predictions\samples


### Error analysis

![confusion matrix](../notebooks/photo/confusion_matrix.png)

Топ-3 найслабші класи на test:
- **bicycle**  -  mAP50  -  0.05
- **people**   -  0.10
- **awning-tricycle**    - 0.13

Збігається з EDA, де рідкісні класи плюс дрібні bbox. Окремо **people** / **pedestrian** плутаються де низький recall у people.

### Good / Bad

**Добре:** великі авто, автобуси на нещільних сценах, люди в крупний план. bbox стабільні, метрики особливо по `car`/`bus` високі;

![photo1](../assets/predictions/good/0000370_00000_d_0000250.jpg)
![photo2](../assets/predictions/good/9999938_00000_d_0000015.jpg)
![photo3](../assets/predictions/good/0000063_02235_d_0000003.jpg)

**Погано:** дрібні пішоходи в натовпі, велосипеди, триколісні, об'єкти в далечині та які плутає за інші. Модель пропускає об'єкти або малює зайве;

![photo4](../assets/predictions/bad/0000152_02214_d_0000150.jpg)
*купа об'єктів пропущено(пішоходи/транспорт)*
![photo5](../assets/predictions/bad/0000250_02350_d_0000001.jpg)
*вдалечині пропущено багато дрібних об'єктів*
![photo6](../assets/predictions/bad/9999938_00000_d_0000464.jpg)
*визначає вікна будівлі як об'єкти*

**Чому:** близько 97.5% bbox маленькі, дисбаланс класів, схожі категорії people/pedestrian.